# Universidad Mayor
# Machine Learning - Trabajo Sumativo Unidad 1

### Predicción de PM2.5 usando variables meteorológicas y contaminantes atmosféricos

#### Integrantes

- Camilo Cáceres

- Nicolás Cruz

- Bruno Yañez

#### Asignatura: Machine Learning

###· Fecha: 24.08.2026

# Introducción

La contaminación atmosférica constituye uno de los principales problemas ambientales a nivel mundial. En este trabajo se utiliza un conjunto de datos de calidad del aire que contiene concentraciones de contaminantes atmosféricos (SO2, NO2 y O3), variables meteorológicas y concentraciones de PM2.5.

El objetivo es desarrollar un pipeline reproducible de Machine Learning para apoyar la predicción de PM2.5, aplicando los conceptos revisados en la Unidad 1: frameworks analíticos, exploración de datos, data wrangling, tratamiento de outliers, análisis de valores faltantes, ventanas de desarrollo y reproducibilidad mediante GitHub.

# 1. Frameworks para Analítica y Machine Learning

## 1. CRISP-DM

### Descripción
Metodología clásica compuesta por:

1. Comprensión del negocio
2. Comprensión de los datos
3. Preparación de datos
4. Modelado
5. Evaluación
6. Despliegue

### Alcances
- Fácil adopción.
- Amplio uso industrial.
- Compatible con cualquier herramienta.

### Limitaciones
- No define control de versiones.
- No incorpora MLOps de forma explícita.

---

## 2. TDSP (Team Data Science Process)

### Descripción
Framework propuesto por Microsoft para proyectos colaborativos.

### Alcances
- Integra Git.
- Integra despliegue.
- Facilita trabajo en equipo.

### Limitaciones
- Más complejo para proyectos pequeños.
- Dependiente de ecosistemas empresariales.

---

## 3. KDD (Knowledge Discovery in Databases)

### Descripción
Framework orientado al descubrimiento de conocimiento.

### Alcances
- Excelente para análisis exploratorio.
- Muy útil para minería de datos.

### Limitaciones
- Menos orientado a producción.
- Menor enfoque en despliegue.

---

## Framework seleccionado

Se selecciona CRISP-DM debido a que:

- El problema es predictivo.
- El dataset es relativamente pequeño (1320 registros).
- El flujo del trabajo encaja perfectamente con las fases solicitadas en la evaluación.
- Facilita la trazabilidad entre exploración, preparación y modelado.

In [ ]:
# Cargamos datos

import pandas as pd
import numpy as np

df = pd.read_csv("datos.csv")

In [ ]:
print(df.shape)

In [ ]:
df.head()

# EDA

# 2. Exploración de Datos

El conjunto de datos contiene:

- SO2: Dióxido de azufre
- NO2: Dióxido de nitrógeno
- O3: Ozono
- Temperature
- Humidity
- Pressure
- WindSpeed
- PM2_5 (variable objetivo)

El problema corresponde a aprendizaje supervisado de regresión, pues existe una variable objetivo continua denominada PM2_5.

In [ ]:
# Obtenemos información general del dataset

df.info()

In [ ]:
df.describe().T

In [ ]:
# Cuantificamos los datos nulos
df.isnull().sum()

### Análisis de Missing Values

La inspección confirma que no existen valores faltantes en ninguna variable.

Por lo tanto:

- No es necesario aplicar imputación.
- No es necesario eliminar registros.
- Se documenta igualmente esta validación por buenas prácticas de Data Wrangling.

In [ ]:
# Buscamos duplicados

df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
#histograma

import seaborn as sns
import matplotlib.pyplot as plt

df.hist(figsize=(14,10))
plt.tight_layout()
plt.show()

In [ ]:
#Heatmap para ver correlaciones

plt.figure(figsize=(8,6))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    cmap="coolwarm"
)

plt.title("Matriz de Correlación")
plt.show()

Las variables con mayor correlación respecto a PM2.5 serán las consideradas potencialmente importantes para el modelado posterior.

In [ ]:
# Para outliers trabajaremos con boxplot (IQR)

fig, axes = plt.subplots(2,4, figsize=(14,8))

for ax, col in zip(axes.flatten(), df.columns):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
def count_outliers_iqr(series):

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3-q1

    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr

    return ((series < lower) | (series > upper)).sum()

for c in df.columns:
    print(c, count_outliers_iqr(df[c]))

In [ ]:
# Aplicamos winsorización para no eliminar información de contaminantes

df_clean = df.copy()

for c in df_clean.columns:

    p1 = df_clean[c].quantile(0.01)
    p99 = df_clean[c].quantile(0.99)

    df_clean[c] = df_clean[c].clip(p1,p99)

Se opta por Winsorización en lugar de eliminación debido a que:

- Los extremos pueden representar episodios reales de contaminación.
- Eliminar registros podría introducir sesgo.
- La Winsorización reduce la influencia de valores extremos sin perder observaciones.

Esta decisión está alineada con las recomendaciones de tratamiento de outliers discutidas en la Unidad 1.

In [ ]:
# Correlacionamos con PM2.5

features = df_clean.drop(columns="PM2_5")

for col in features.columns:

    plt.figure(figsize=(5,3))

    sns.scatterplot(
        data=df_clean,
        x=col,
        y="PM2_5"
    )

    plt.show()

### Hold-Out

70% entrenamiento
15% validación
15% test

Ventajas:
- Simple
- Rápido

Desventajas:
- Dependencia de una sola partición

### Train-Test Split

80% entrenamiento
20% test

Ventajas:
- Fácil implementación

Desventajas:
- Sin conjunto explícito de validación

### K-Fold Cross Validation

Ventajas:
- Mejor aprovechamiento de datos.
- Evaluación más robusta.

Desventajas:
- Mayor costo computacional.

Se selecciona Hold-Out 70/15/15.

Justificación:

- Dataset de tamaño moderado (1320 observaciones).
- Permite disponer simultáneamente de entrenamiento, validación y test.
- Facilita futuras etapas de ajuste de hiperparámetros.

In [ ]:
# Aplicamos la elección

from sklearn.model_selection import train_test_split

X = df_clean.drop(columns="PM2_5")
y = df_clean["PM2_5"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

Estructura propuesta para GitHUB

AirQuality_PM25/
│
├── data/
│   └── datos.csv
│
├── notebooks/
│   └── unidad1_air_quality.ipynb
│
├── requirements.txt
│
├── README.md
│
└── .gitignore

readme.md

# Air Quality PM2.5 Prediction

## Objetivo

Desarrollar un pipeline reproducible de Machine Learning para estimar PM2.5 utilizando variables meteorológicas y contaminantes atmosféricos.

## Dataset

Air Quality Dataset

## Variables

- SO2
- NO2
- O3
- Temperature
- Humidity
- Pressure
- WindSpeed

Target:
- PM2_5

## Requisitos

```bash
pip install -r requirements.txt
```

## Ejecución

Abrir notebook:

```bash
jupyter notebook
```

## Reproducibilidad

Python 3.11
Pandas
NumPy
Matplotlib
Seaborn
Scikit-Learn

requirements:

pandas
numpy
matplotlib
seaborn
scikit-learn
jupyter

# Conclusiones

- Se identificó que el problema corresponde a aprendizaje supervisado de regresión.
- Se compararon los frameworks CRISP-DM, TDSP y KDD, seleccionando CRISP-DM por su adaptación al problema.
- El EDA permitió comprender el comportamiento de las variables de contaminación atmosférica y meteorológicas.
- No se detectaron valores faltantes.
- Los outliers fueron tratados mediante Winsorización para preservar información relevante.
- Se evaluaron distintos métodos de partición de datos y se seleccionó Hold-Out 70/15/15.
- Se definió una estructura reproducible mediante GitHub, README y requirements.txt.

El trabajo cumple los principios de preparación de datos, validación y reproducibilidad revisados en la Unidad 1.